# Banking Portfolio Risk Analysis - 01: Data Cleaning & Preprocessing
**Author:** Business Analyst Portfolio Project  
**Project Scope:** Banking Domain Risk Analytics & Credit Portfolio Slicing  
**Pipeline Stage:** Notebook 1 of 2 (Strictly Sequential Pipeline)  

---

### Business Objective
The objective of this project is to develop risk analytics across a portfolio of **3,000 retail, commercial, and private banking clients** to identify exposure concentrations, analyze credit card and business loan utilization, and prepare curated data for BI dashboards and risk underwriting models.

### Notebook 1 Deliverables:
1. **Data Ingestion & Schema Audit:** Validate 25 raw banking attributes.
2. **Data Integrity & Quality Check:** Audit missing values, duplicate entries, and unique identifiers.
3. **Type Casting:** Parse `Joined Bank` to standard `datetime64` and calculate customer tenure / engagement days.
4. **Feature Engineering:**
   - Continuous income binning into `Income Band` (`Low`, `Med`, `High`).
   - Categorical dimension mappings (`BRId` -> Relationship type, `GenderId` -> Gender).
   - Aggregate financial KPIs (`Total Loan`, `Total Deposit`, `Total Fees`) aligned with business requirements.
5. **Production Export:** Save the cleaned, standardized dataset to `../data/processed/banking_cleaned.csv` for sequential ingestion by `02_exploratory_data_analysis.ipynb`.

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Robust relative path to load raw dataset
raw_data_path = os.path.join("..", "data", "raw", "Banking.csv")
if not os.path.exists(raw_data_path):
    # Fallback paths for root directory or datasets/ subfolder
    if os.path.exists("Banking.csv"):
        raw_data_path = "Banking.csv"
    elif os.path.exists(os.path.join("datasets", "Banking.csv")):
        raw_data_path = os.path.join("datasets", "Banking.csv")
    elif os.path.exists("Banking.xlsx"):
        raw_data_path = "Banking.xlsx"

print(f"Loading raw banking dataset from: {raw_data_path}")
if raw_data_path.endswith(".xlsx"):
    df = pd.read_excel(raw_data_path)
else:
    if not os.path.isfile(raw_data_path):
        raise FileNotFoundError(
            f"Banking.csv was not found. Expected location: "
            f"{os.path.abspath(raw_data_path)}. Place the file there or update "
            "`raw_data_path` to its actual location."
        )

    df = pd.read_csv(raw_data_path)

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\HP\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\HP\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\HP\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\HP\anaconda3\Lib\site-packages\tornado

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Loading raw banking dataset from: ..\data\raw\Banking.csv


FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\Banking.csv'

## 1. Initial Data Inspection
We inspect the first 5 records of the dataset to verify column alignments and baseline record structure.

In [ ]:
df.head()

## 2. Schema and Data Type Inspection
We review the technical schema across all 25 raw fields. The dataset contains:
- **Client Demographics:** `Client ID`, `Name`, `Age`, `Location ID`, `Nationality`, `Occupation`, `GenderId`.
- **Relationship Metadata:** `Joined Bank`, `Banking Contact`, `Fee Structure`, `Loyalty Classification`, `BRId`, `IAId`.
- **Financial Balances:** `Estimated Income`, `Superannuation Savings`, `Amount of Credit Cards`, `Credit Card Balance`, `Bank Loans`, `Bank Deposits`, `Checking Accounts`, `Saving Accounts`, `Foreign Currency Account`, `Business Lending`.
- **Risk & Asset Indicators:** `Properties Owned`, `Risk Weighting`.

In [ ]:
df.info()

## 3. Data Integrity & Completeness Audit
We verify that there are no missing values (`NaN`) or duplicate account records, and confirm that `Client ID` serves as a strict primary key.

In [ ]:
# Audit missing values
null_counts = df.isnull().sum()
print("--- Missing Values Count per Column ---")
print(null_counts[null_counts > 0] if (null_counts > 0).any() else "Zero missing values detected across all columns.")

# Duplicate records audit
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows detected: {duplicate_count}")

# Primary key verification
unique_clients = df['Client ID'].nunique()
print(f"Total unique Client IDs: {unique_clients} (Matches row count: {unique_clients == len(df)})")

## 4. Descriptive Statistics & Baseline Ranges
We run descriptive statistics to establish the distribution boundaries and check for abnormal outliers across all numeric variables.

In [ ]:
df.describe().T

**Key Baseline Observations:**
- **Age:** Spans from 17 to 85 years (mean 51.04, median 51.00), showing balanced representation across career stages.
- **Estimated Income:** Ranges from $15,919.48 to $522,330.26 (mean $171,305.03, median $142,313.48).
- **Credit Cards:** 1 to 3 cards per customer, with average holding at 1.46 cards.
- **Credit Card Balance:** Averages $3,176.21 with a maximum of $13,991.99.
- **Bank Loans & Business Lending:** Display substantial right-skewed maximums ($2.67M and $3.83M, respectively).
- **Risk Weighting:** Formally bounded as an ordinal score from 1 (lowest risk) to 5 (highest risk).

## 5. Temporal Formatting & Customer Tenure Engineering
The `Joined Bank` column is stored as an object string. We parse it into a standard `datetime64[ns]` timestamp and engineer:
- `Year Joined`: Joining year for cohort analysis.
- `Engagement Days`: Total days of customer relationship relative to a fixed baseline date (2026-01-01).

In [ ]:
# Parse Joined Bank to datetime format
df['Joined Bank'] = pd.to_datetime(df['Joined Bank'], format='%d-%m-%Y', errors='coerce')
if df['Joined Bank'].isnull().any():
    # Fallback parser for alternative date formats
    df['Joined Bank'] = pd.to_datetime(df['Joined Bank'], format='mixed', errors='coerce')

# Benchmark reference date aligned with project timeframe
benchmark_date = pd.to_datetime('2026-01-01')
df['Year Joined'] = df['Joined Bank'].dt.year
df['Engagement Days'] = (benchmark_date - df['Joined Bank']).dt.days

print("Temporal transformations completed:")
df[['Client ID', 'Joined Bank', 'Year Joined', 'Engagement Days']].head()

## 6. Business Logic & Feature Engineering
We construct key business features aligned with the Business Requirements Document and Power BI DAX formulas:

1. **Income Band:** Segmenting clients into 3 strata:
   - `Low`: Income < $100,000
   - `Med`: $100,000 <= Income < $300,000
   - `High`: Income >= $300,000
2. **Banking Relationship Mapping:** Converting numeric `BRId` into human-readable business units:
   - 1: `Retail`
   - 2: `Institutional`
   - 3: `Private Bank`
   - 4: `Commercial`
3. **Gender Identity Mapping:**
   - 1: `Male`
   - 2: `Female`
4. **Total Loan Exposure:** Aggregate debt liability:
   $$\text{Total Loan} = \text{Bank Loans} + \text{Business Lending} + \text{Credit Card Balance}$$
5. **Total Deposit Liquidity:** Aggregate customer funds:
   $$\text{Total Deposit} = \text{Bank Deposits} + \text{Saving Accounts} + \text{Checking Accounts} + \text{Foreign Currency Account}$$
6. **Processing Fee Calculation:** Account service fee based on tier:
   $$\text{Total Fees} = \text{Total Loan} \times \text{Processing Fee Rate}$$
   - `High` fee structure: 5.0% (0.05)
   - `Mid` fee structure: 3.0% (0.03)
   - `Low` fee structure: 1.5% (0.015)

In [ ]:
# 1. Income Band segmentation
bins = [0, 100000, 300000, float('inf')]
labels = ['Low', 'Med', 'High']
df['Income Band'] = pd.cut(df['Estimated Income'], bins=bins, labels=labels, right=False)

# 2. Relationship and Demographic Mapping
br_map = {1: 'Retail', 2: 'Institutional', 3: 'Private Bank', 4: 'Commercial'}
df['Banking Relationship'] = df['BRId'].map(br_map)

gender_map = {1: 'Male', 2: 'Female'}
df['Gender'] = df['GenderId'].map(gender_map)

# 3. Total Loan & Total Deposit Aggregations
df['Total Loan'] = df['Bank Loans'] + df['Business Lending'] + df['Credit Card Balance']
df['Total Deposit'] = (
    df['Bank Deposits'] +
    df['Saving Accounts'] +
    df['Checking Accounts'] +
    df['Foreign Currency Account']
)

# 4. Processing Fee Calculation
fee_rate_map = {'High': 0.05, 'Mid': 0.03, 'Low': 0.015}
df['Processing Fee Rate'] = df['Fee Structure'].map(fee_rate_map).fillna(0.02)
df['Total Fees'] = df['Total Loan'] * df['Processing Fee Rate']

print("Feature engineering completed successfully.")
df[['Client ID', 'Income Band', 'Banking Relationship', 'Gender', 'Total Loan', 'Total Deposit', 'Total Fees']].head()

## 7. Portfolio Metric Validation
We validate summary totals and frequency distributions of the newly engineered fields against project benchmarks.

In [ ]:
print("=== Income Band Distribution ===")
print(df['Income Band'].value_counts())

print("\n=== Banking Relationship Breakdown ===")
print(df['Banking Relationship'].value_counts())

print("\n=== Portfolio Macro Financial Totals ===")
print(f"Total Portfolio Lending Exposure: ${df['Total Loan'].sum():,.2f}")
print(f"Total Portfolio Deposit Liquidity: ${df['Total Deposit'].sum():,.2f}")
print(f"Total Projected Fee Revenue:      ${df['Total Fees'].sum():,.2f}")

## 8. Export Processed Dataset for Downstream Pipeline
We save the curated dataframe to `../data/processed/banking_cleaned.csv`.
The downstream notebook (`02_exploratory_data_analysis.ipynb`) will load this file directly, ensuring a strict, non-redundant sequential workflow.

In [ ]:
# Create processed data directory if it does not exist
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "banking_cleaned.csv")

# Export cleaned dataset
df.to_csv(output_path, index=False)

# Also save locally in current directory for standalone reproducibility
df.to_csv("banking_cleaned.csv", index=False)

print(f"Cleaned dataset exported to: {output_path}")
print(f"Final record count: {df.shape[0]} rows, {df.shape[1]} columns.")

### Pipeline Status
✅ **Notebook 01 (Data Cleaning & Preprocessing) is complete.**  
➡️ **Next Step:** Open and run `02_exploratory_data_analysis.ipynb` for univariate distributions, bivariate nationality cuts, balance probability density curves, and correlation heatmaps.